# Choose a FAISS search mode

[All How-to Guides](README.md)

Compare exact, approximate, and automatic search on the same dataset. Check how each affects the selected neighbors and the ATE.

## 1. Prepare a shared input

Use one numeric baseline feature. Keep Mahalanobis distance and the number of neighbors fixed across runs.

The generator defaults to `dependent_division=True`. The example spells it out to show that assignment depends on the baseline feature.

In [1]:
import pandas as pd

from hypex import Matching
from hypex.dataset import Dataset, FeatureRole, TargetRole, TreatmentRole
from hypex.utils.tutorial_data_creation import gen_control_variates_df

frame = gen_control_variates_df(
    data_size=8000, dependent_division=True, random_state=42
)
data = Dataset(
    roles={
        "Treatment": TreatmentRole(),
        "Target": TargetRole(),
        "X_lag_1": FeatureRole(),
    },
    data=frame[["Treatment", "Target", "X_lag_1"]].copy(),
)
data.shape

(8000, 3)

Each row represents one user. The search uses `X_lag_1`.

## 2. Compare each mode with exact search

`base` performs exact search. `fast` uses approximate IVF search. `auto` chooses a mode based on the data size. Use `base` as the reference and retrieve `indexes` to compare the neighbors.

Only `faiss_mode` changes between runs. This version requires both `compute_indexes=True` and `extract_full_data=True` to populate `indexes`. Leave them at their defaults if you only need `summary`. `quality_tests=[]` disables the balance checks so this example can focus on neighbor assignments and ATE.

`Matching` defaults to `distance="mahalanobis"` and `faiss_mode="auto"`. The example spells out the shared distance and includes `auto` as one of the modes under comparison.

Read the ATE from `summary` with `get_values()` and align neighbor indices by input row before comparing them.

In [2]:
results = {
    mode: Matching(
        distance="mahalanobis",
        faiss_mode=mode,
        quality_tests=[],
        compute_indexes=True,
        extract_full_data=True,
    ).execute(data)
    for mode in ("base", "fast", "auto")
}
exact_indexes = results["base"].indexes.sort().to_numpy()
comparison = pd.DataFrame.from_dict(
    {
        mode: {
            "ATE": result.summary.get_values(row="ATE", column="Effect Size"),
            "different neighbors (%)": 100
            * (exact_indexes != result.indexes.sort().to_numpy()).mean(),
        }
        for mode, result in results.items()
    },
    orient="index",
)
comparison.round(3)

,ATE,different neighbors (%)
base,7.452,0.000
fast,6.764,10.725
auto,7.452,0.000


On this sample, `auto` agreed with exact search. `fast` can change the matched pairs and the ATE. For your own data, compare runtime at the expected scale, match quality, and estimate stability.

The [in-depth Matching tutorial](../quick_tutorials/Matching.ipynb) covers distance and neighbor selection.